# Winter Storm Fern: Predicting Neighborhood Demand Swings When Solar Goes Dark

This notebook reproduces the full analysis from the [ODS-E blog article](https://opendataschema.energy/articles/winter-storm-fern-solar-whiteout-demand-forecasting.html).

**What you'll do:**
1. Load raw inverter data from a fictional 5 MW solar farm (SMA Sunny Portal export)
2. Transform it to the ODS-E energy-timeseries schema
3. Visualize the production collapse during the storm
4. Download NREL ResStock building consumption data
5. Profile heating demand by building vintage
6. Calculate the demand swing when solar goes dark
7. Map storm vulnerability by neighborhood using Census PUMA boundaries

**Self-contained:** The first cell installs all dependencies. No prior setup required.

**Data sources:**
- **Solar production:** Bundled SMA Sunny Portal CSV (`bayou_solar_farm_sma_export.csv`) — synthetic but realistic
- **Building consumption:** NREL ResStock 2024 R2 baseline (downloaded from OEDI S3 — free, no credentials)
- **Geographic boundaries:** Census TIGER 2020 PUMA shapefile (downloaded from Census Bureau)

In [ ]:
# Install dependencies (safe to re-run — pip skips already-installed packages)
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "matplotlib", "geopandas", "requests"
])
print("All dependencies installed.")

In [ ]:
import os
import json
import zipfile
import io

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import geopandas as gpd
import requests

# Style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#cccccc',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#cccccc',
})

BLUE = '#455BF1'
RED = '#E20419'
DARK = '#1a1a2e'
GREY = '#6b7280'
AMBER = '#f59e0b'
GREEN = '#10b981'

# Constants
HARRIS_COUNTY = 'G4802010'
STORM_START = '2026-01-24'
STORM_END = '2026-01-27'

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(NOTEBOOK_DIR, '_data')
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Data cache: {DATA_DIR}")

---
## Step 1: Load the Solar Farm Data

You're a utility analyst. Your grid planning team just received a data export from **Bayou Solar Farm** — a fictional 5 MW ground-mounted installation in eastern Harris County equipped with SMA Sunny Central inverters.

The bundled CSV (`bayou_solar_farm_sma_export.csv`) is structured exactly like a real SMA Sunny Portal export:
- `Timestamp` — 5-minute intervals
- `Total Power (kW)` — instantaneous AC output
- `Grid Frequency (Hz)` — grid frequency at the point of interconnection
- `Status` — inverter state: `Ok`, `Warning`, `Disturbance`, or `Off`
- `Daily Yield (kWh)` — cumulative energy produced since midnight, resets each day

In [ ]:
sma_csv = os.path.join(NOTEBOOK_DIR, 'bayou_solar_farm_sma_export.csv')
sma_df = pd.read_csv(sma_csv)
sma_df['Timestamp'] = pd.to_datetime(sma_df['Timestamp'])

print(f"Rows: {len(sma_df):,}")
print(f"Date range: {sma_df['Timestamp'].min()} to {sma_df['Timestamp'].max()}")
print(f"Columns: {list(sma_df.columns)}")
print()

# Show a few normal-day rows and a few storm-day rows
print("Normal day (Jan 20, midday):")
display(sma_df[(sma_df['Timestamp'].dt.date == pd.Timestamp('2026-01-20').date()) &
               (sma_df['Timestamp'].dt.hour.between(11, 13))].head(6))

print("\nStorm day (Jan 25, midday):")
display(sma_df[(sma_df['Timestamp'].dt.date == pd.Timestamp('2026-01-25').date()) &
               (sma_df['Timestamp'].dt.hour.between(11, 13))].head(6))

---
## Step 2: Transform to ODS-E

Raw SMA exports can't be directly joined with building consumption data — the column names, energy semantics, and status codes are all vendor-specific. ODS-E normalizes heterogeneous inverter feeds into a common [energy-timeseries](https://opendataschema.energy/docs/schemas/energy-timeseries) schema.

The transform maps:
- `Total Power (kW)` × 5/60 → `kWh` (energy per interval)
- `Status` → `error_type` enum (`Ok`→`normal`, `Warning`→`warning`, `Disturbance`→`critical`, `Off`→`normal`)
- Adds `direction: "generation"` and `asset_id: "BAYOU-SOLAR-001"`

In [ ]:
# Transform SMA export to ODS-E energy-timeseries records
error_map = {'Ok': 'normal', 'Warning': 'warning', 'Disturbance': 'critical', 'Off': 'normal'}

odse_records = pd.DataFrame({
    'timestamp': sma_df['Timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S-06:00'),
    'kWh': (sma_df['Total Power (kW)'] * 5 / 60).round(3),
    'error_type': sma_df['Status'].map(error_map),
    'direction': 'generation',
    'asset_id': 'BAYOU-SOLAR-001',
})

print(f"Transformed {len(odse_records):,} records")
print(f"Total generation: {odse_records['kWh'].sum():,.1f} kWh")
print()

# Show a normal record and a storm record as JSON
normal_idx = odse_records[(odse_records['kWh'] > 200) & (odse_records['error_type'] == 'normal')].index[0]
storm_idx = odse_records[odse_records['error_type'] == 'critical'].index[0]

print("Normal production record (clear day):")
print(json.dumps(odse_records.loc[normal_idx].to_dict(), indent=2))
print()
print("Storm record (disturbance):")
print(json.dumps(odse_records.loc[storm_idx].to_dict(), indent=2))
print()

# ODS-E asset-metadata for the farm
asset_metadata = {
    'asset_id': 'BAYOU-SOLAR-001',
    'asset_type': 'solar_pv',
    'capacity_kw': 5000,
    'site_id': 'BAYOU-FARM',
    'location': {
        'latitude': 29.78,
        'longitude': -95.18,
        'county': 'Harris County',
        'state': 'TX',
        'climate_zone': 'IECC 2A',
    },
    'inverter': 'SMA Sunny Central',
    'mounting': 'ground-mount, single-axis tracker',
}
print("Asset metadata:")
print(json.dumps(asset_metadata, indent=2))

---
## Step 3: Visualize the Storm Collapse

Aggregate daily production from the ODS-E records and plot the storm window. On normal January days, Bayou Solar Farm produces ~21 MWh/day. During Fern (Jan 24–27), production collapses — the farm effectively goes dark.

In [ ]:
# Aggregate daily production from ODS-E records
odse_records['date'] = pd.to_datetime(odse_records['timestamp']).dt.date
daily_prod = odse_records.groupby('date')['kWh'].sum() / 1000  # MWh

storm_days = pd.date_range(STORM_START, STORM_END, freq='D')
dates = daily_prod.index
daily_mwh = daily_prod.values

fig, ax = plt.subplots(figsize=(11, 6))

colors = [RED if pd.Timestamp(d) in storm_days else BLUE for d in dates]
bars = ax.bar(range(len(dates)), daily_mwh, color=colors, width=0.7,
              edgecolor='white', linewidth=0.5)

date_labels = [pd.Timestamp(d).strftime('%b %d') for d in dates]
ax.set_xticks(range(len(dates)))
ax.set_xticklabels(date_labels, fontsize=10)
ax.set_ylabel('Daily Production (MWh)')
ax.set_title('Bayou Solar Farm \u2014 Daily Production During Winter Storm Fern\n'
             '5 MW Ground-Mount, Harris County TX \u2014 SMA Sunny Central Inverters',
             fontsize=13)

# Value labels
for i, (bar, val) in enumerate(zip(bars, daily_mwh)):
    is_storm = pd.Timestamp(dates[i]) in storm_days
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9,
            fontweight='bold' if is_storm else 'normal',
            color=RED if is_storm else DARK)

# Storm annotation
ax.axvspan(3.5, 7.5, alpha=0.08, color=RED, zorder=0)
ax.annotate('Winter Storm Fern\nJan 24\u201327',
            xy=(5.5, max(daily_mwh) * 0.85), fontsize=10, color=RED,
            fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor=RED, alpha=0.9))

# Normal avg reference line
normal_mask = [pd.Timestamp(d) not in storm_days for d in dates]
normal_avg = np.mean(daily_mwh[normal_mask])
ax.axhline(y=normal_avg, color=GREY, linestyle='--', alpha=0.5, linewidth=1)
ax.text(len(dates) - 0.5, normal_avg + 0.5,
        f'Normal avg: {normal_avg:.1f} MWh/day', fontsize=9, color=GREY, ha='right')

# Drop percentage
storm_mask = [pd.Timestamp(d) in storm_days for d in dates]
storm_avg = np.mean(daily_mwh[storm_mask])
drop_pct = (1 - storm_avg / normal_avg) * 100
ax.annotate(f'{drop_pct:.0f}% production\ndrop during storm',
            xy=(5, storm_avg), xytext=(8, 8),
            fontsize=9, color=RED, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor=RED, alpha=0.9))

ax.set_ylim(0, max(daily_mwh) * 1.2)
plt.tight_layout()
plt.show()

print(f"\nNormal daily avg: {normal_avg:.1f} MWh")
print(f"Storm daily avg:  {storm_avg:.1f} MWh")
print(f"Production drop:  {drop_pct:.0f}%")

---
## Step 4: Download ResStock Building Data

[NREL ResStock](https://data.openei.org/submissions/4520) provides physics-based energy consumption simulations for every residential building type in the US. We'll download the Texas baseline from the OEDI S3 bucket (free, no credentials required) and filter to Harris County.

This is a ~130 MB download. The notebook caches the file locally so subsequent runs skip the download.

In [ ]:
RESSTOCK_URL = (
    'https://oedi-data-lake.s3.amazonaws.com/'
    'nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/'
    '2024/resstock_tmy3_release_2/metadata_and_annual_results/'
    'by_state/state=TX/csv/'
    'TX_baseline_metadata_and_annual_results.csv'
)

resstock_path = os.path.join(DATA_DIR, 'tx_resstock_baseline.csv')

if os.path.exists(resstock_path):
    print(f"Using cached file: {resstock_path}")
else:
    print(f"Downloading Texas ResStock baseline (~130 MB)...")
    resp = requests.get(RESSTOCK_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    downloaded = 0
    with open(resstock_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"  {downloaded / 1e6:.0f} / {total / 1e6:.0f} MB ({pct:.0f}%)", end='\r')
    print(f"\n  Saved to {resstock_path}")

df = pd.read_csv(resstock_path, low_memory=False)
print(f"\nTexas total: {len(df):,} building models, {df['weight'].sum():,.0f} weighted dwellings")

# Filter to Harris County
harris = df[df['in.county'] == HARRIS_COUNTY].copy()
print(f"Harris County: {len(harris):,} models, {harris['weight'].sum():,.0f} dwellings")

---
## Step 5: Heating Demand by Construction Era

During a winter storm, older buildings draw dramatically more heating power. ResStock lets us quantify this by construction vintage.

In [ ]:
heating_col = 'out.electricity.heating.energy_consumption.kwh'

vintage_heating = (
    harris.groupby('in.vintage')
    .apply(lambda g: (g[heating_col] * g['weight']).sum() / g['weight'].sum(),
           include_groups=False)
    .sort_index()
)

vintage_order = sorted(harris['in.vintage'].unique())
vintage_heating = vintage_heating.reindex(vintage_order)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [RED if v >= 1500 else BLUE for v in vintage_heating.values]
bars = ax.barh(range(len(vintage_heating)), vintage_heating.values, color=colors, height=0.7)

ax.set_yticks(range(len(vintage_heating)))
ax.set_yticklabels(vintage_heating.index, fontsize=10)
ax.set_xlabel('Weighted Average Heating Electricity (kWh/dwelling/year)')
ax.set_title('Heating Electricity Demand by Construction Era\n'
             'Harris County, TX \u2014 ResStock 2024 R2 Baseline')
ax.invert_yaxis()

for i, (bar, val) in enumerate(zip(bars, vintage_heating.values)):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=9, color=DARK)

ax.axvline(x=vintage_heating.median(), color=GREY, linestyle='--', alpha=0.6, linewidth=1)
ax.text(vintage_heating.median() + 20, len(vintage_heating) - 0.5,
        f'Median: {vintage_heating.median():,.0f} kWh', fontsize=9, color=GREY)

plt.tight_layout()
plt.show()

print('\nHeating demand by vintage:')
for vintage, kwh in vintage_heating.items():
    print(f'  {vintage:<20} {kwh:>8,.0f} kWh')

---
## Step 6: Calculate the Demand Swing

When Bayou Solar Farm's production collapses during a storm, the neighborhoods it serves lose their solar offset and the grid sees their full building consumption. The demand swing equals the farm's lost production.

We model this at different PV penetration levels (5%, 10%, 20%) for the top 10 Harris County PUMAs by energy consumption.

In [ ]:
energy_col = 'out.site_energy.total.energy_consumption.kwh'
pv_col = 'out.electricity.pv.energy_consumption.kwh'
puma_col = 'in.puma'

# Average PV generation per PV-equipped home (negative = generation)
pv_homes = harris[harris['in.has_pv'] == 'Yes']
avg_pv_gen_kwh = abs((pv_homes[pv_col] * pv_homes['weight']).sum() / pv_homes['weight'].sum())
print(f'Avg PV generation per home: {avg_pv_gen_kwh:,.0f} kWh/yr')

# Per-PUMA total energy
puma_stats = harris.groupby(puma_col).agg(
    total_energy=pd.NamedAgg(column=energy_col,
                             aggfunc=lambda x: (x * harris.loc[x.index, 'weight']).sum()),
    total_dwellings=pd.NamedAgg(column='weight', aggfunc='sum')
).sort_values('total_energy', ascending=False).head(10)

# Demand swing at different penetration levels
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(puma_stats))
width = 0.25

for i, (pct, color) in enumerate([(5, BLUE), (10, RED), (20, GREEN)]):
    swing_gwh = (puma_stats['total_dwellings'] * (pct/100) * avg_pv_gen_kwh) / 1_000_000
    ax.bar(x + i*width, swing_gwh.values, width, label=f'{pct}% PV penetration',
           color=color, alpha=0.85)

ax.set_xlabel('PUMA (Public Use Microdata Area)')
ax.set_ylabel('Annual Demand Swing When Solar Goes Dark (GWh)')
ax.set_title('Predicted Demand Swing by Neighborhood at Projected Solar Penetration\n'
             'Harris County \u2014 Top 10 PUMAs by Energy Consumption')
ax.set_xticks(x + width)
ax.set_xticklabels([f'PUMA\n{p}' for p in puma_stats.index], fontsize=8)
ax.legend(fontsize=10)

ax.annotate('Higher bars = bigger demand surprise\nwhen solar drops to zero',
            xy=(0.98, 0.95), xycoords='axes fraction',
            fontsize=9, color=GREY, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#e0e0e0'))

plt.tight_layout()
plt.show()

---
## Step 7: Map Storm Vulnerability by Neighborhood

Combine four risk factors into a composite vulnerability score per PUMA, then map it onto Census TIGER 2020 PUMA boundaries to see **where** storm demand risk concentrates geographically.

**Composite = Heating Intensity (30%) + Dwelling Density (25%) + Pre-1980 Stock (25%) + Electric Heating (20%)**

The Census TIGER shapefile (~5 MB) is downloaded and cached automatically.

In [ ]:
PUMA_URL = 'https://www2.census.gov/geo/tiger/TIGER2022/PUMA/tl_2022_48_puma20.zip'
puma_zip_path = os.path.join(DATA_DIR, 'tl_2022_48_puma20.zip')
puma_shp_dir = os.path.join(DATA_DIR, 'tl_2022_48_puma20')
puma_shp_path = os.path.join(puma_shp_dir, 'tl_2022_48_puma20.shp')

if not os.path.exists(puma_shp_path):
    print('Downloading Census TIGER 2020 PUMA shapefile (~5 MB)...')
    resp = requests.get(PUMA_URL)
    resp.raise_for_status()
    with open(puma_zip_path, 'wb') as f:
        f.write(resp.content)
    with zipfile.ZipFile(puma_zip_path, 'r') as z:
        z.extractall(puma_shp_dir)
    print(f'  Extracted to {puma_shp_dir}')
else:
    print(f'Using cached shapefile: {puma_shp_path}')

# Compute per-PUMA vulnerability factors
def compute_puma_risk(g):
    total_w = g['weight'].sum()
    if total_w == 0:
        return pd.Series({'heating_intensity': 0, 'dwelling_count': 0,
                          'pre_1980_pct': 0, 'electric_heat_pct': 0})

    heating_int = (g[heating_col] * g['weight']).sum() / total_w

    pre_1980_vintages = [v for v in g['in.vintage'].unique()
                         if any(era in str(v) for era in
                                ['<1940', '1940s', '1950s', '1960s', '1970s'])]
    pre_1980 = g[g['in.vintage'].isin(pre_1980_vintages)]['weight'].sum() / total_w * 100

    elec_heat = g[g['in.hvac_heating_type_and_fuel'].str.contains(
        'Electric', na=False)]['weight'].sum() / total_w * 100

    return pd.Series({
        'heating_intensity': heating_int,
        'dwelling_count': total_w,
        'pre_1980_pct': pre_1980,
        'electric_heat_pct': elec_heat,
    })

puma_risk = harris.groupby(puma_col).apply(compute_puma_risk, include_groups=False)
print(f'Computed risk for {len(puma_risk)} PUMAs')

# Normalize to 0-1
puma_risk_norm = puma_risk.copy()
for col in puma_risk_norm.columns:
    lo, hi = puma_risk_norm[col].min(), puma_risk_norm[col].max()
    puma_risk_norm[col] = (puma_risk_norm[col] - lo) / (hi - lo) if hi > lo else 0.5

# Composite vulnerability score
puma_risk_norm['vulnerability'] = (
    puma_risk_norm['heating_intensity'] * 0.30 +
    puma_risk_norm['dwelling_count'] * 0.25 +
    puma_risk_norm['pre_1980_pct'] * 0.25 +
    puma_risk_norm['electric_heat_pct'] * 0.20
)

# Map ResStock PUMA codes to Census PUMACE20
puma_risk_norm['PUMACE20'] = puma_risk_norm.index.str.replace('G480', '', regex=False)
puma_risk['PUMACE20'] = puma_risk.index.str.replace('G480', '', regex=False)

# Load shapefile and join
gdf = gpd.read_file(puma_shp_path)
harris_codes = puma_risk_norm['PUMACE20'].tolist()
harris_gdf = gdf[gdf['PUMACE20'].isin(harris_codes)].copy()
harris_gdf = harris_gdf.merge(
    puma_risk_norm[['PUMACE20', 'vulnerability']], on='PUMACE20', how='left')
harris_gdf = harris_gdf.merge(
    puma_risk[['PUMACE20', 'heating_intensity', 'dwelling_count']], on='PUMACE20', how='left')

print(f'Matched {len(harris_gdf)} PUMAs in shapefile')

# Choropleth
fig, ax = plt.subplots(figsize=(12, 10))

cmap = LinearSegmentedColormap.from_list(
    'vuln', ['#e8eef7', '#455BF1', '#b91c4c', '#E20419'], N=256)

harris_gdf.plot(
    column='vulnerability', cmap=cmap, ax=ax, legend=True,
    legend_kwds={'label': 'Storm Demand Vulnerability Score',
                 'orientation': 'horizontal', 'shrink': 0.6,
                 'pad': 0.02, 'aspect': 30},
    edgecolor='white', linewidth=0.8, vmin=0, vmax=1)

# Label top 5 most vulnerable PUMAs
top5 = harris_gdf.nlargest(5, 'vulnerability')
for _, row in top5.iterrows():
    centroid = row.geometry.centroid
    name = row['NAMELSAD20']
    if '--' in name:
        name = name.split('--')[1].strip()
    name = name.replace(' PUMA', '')
    if len(name) > 35:
        name = name[:32] + '...'
    ax.annotate(
        f"{name}\n(score: {row['vulnerability']:.2f})",
        xy=(centroid.x, centroid.y),
        fontsize=7.5, fontweight='bold', color=DARK,
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                  edgecolor='#cccccc', alpha=0.85))

ax.set_title(
    'Storm Demand Vulnerability by Neighborhood\n'
    'Harris County, TX \u2014 PUMAs colored by composite risk score\n'
    'Source: ResStock 2024 R2 + Census TIGER 2020 PUMA Boundaries',
    fontsize=13, fontweight='bold', pad=15)
ax.axis('off')

fig.text(0.02, 0.02,
         'Composite = Heating Intensity (30%) + Dwelling Density (25%) '
         '+ Pre-1980 Stock (25%) + Electric Heating (20%)',
         fontsize=8, color=GREY, style='italic')

plt.tight_layout()
plt.show()

print('\nTop 5 most vulnerable PUMAs:')
for _, row in top5.iterrows():
    print(f"  {row['NAMELSAD20'][:55]:<55}  score={row['vulnerability']:.2f}")

---
## Summary

This analysis combined two data sources that most utilities keep separate:

1. **Solar production data** (SMA inverter export) normalized through ODS-E into a standard schema
2. **Building consumption data** (NREL ResStock) at neighborhood resolution

Joining them reveals where storm demand spikes hit hardest when solar generation collapses — a question no single data source can answer alone.

**Next steps:**
- [ODS-E Schema Docs](https://opendataschema.energy/docs/schemas/overview) — Full schema reference
- [Municipal Emissions Attribution](https://opendataschema.energy/docs/patterns/municipal-emissions-attribution) — Disaggregate CO\u2082 by building sector
- [Cross-City Benchmarking](https://opendataschema.energy/docs/patterns/cross-city-benchmarking) — Compare cities using the same methodology
- [ComStock & ResStock Guide](https://opendataschema.energy/docs/building-integration/comstock-resstock) — Deep dive into the building datasets
- [ODS-E on GitHub](https://github.com/AsobaCloud/odse) — Schema definitions, transforms, and validation tools